# 阿里 PPU：DolphinDB 分钟数据全量内存训练

本 Notebook 从 DolphinDB 按交易日并发读取分钟数据，直接写入 PPU 节点 RAM。训练开始后不再访问 DolphinDB，也不生成分钟 MemMap 文件。700GB 节点默认预留 100GB，基础分钟数组上限设为 550GB。

In [ ]:
from pathlib import Path
import json
import os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    raise FileNotFoundError('请先进入 AlphaMining-GFlowNet-AlphaEval 仓库根目录')
print('PROJECT_ROOT =', PROJECT_ROOT)

## 安装依赖

安装完成后如 Jupyter 提示重启 Kernel，请重启后从下一单元格继续。

In [ ]:
%pip install -q -r requirements.txt
%pip install -q -r requirements-ddb.txt

## 检查 DDB 环境变量

请在启动 JupyterLab 前设置这些环境变量。Notebook 只检查是否存在，不会打印账号或密码。

In [ ]:
RAM_CACHE_DIR = Path(os.environ.get(
    'ALPHAMINING_RAM_CACHE_DIR', 'results/minute_ppu_ddb_ram/ram_cache'
)).resolve()
os.environ['ALPHAMINING_RAM_CACHE_DIR'] = str(RAM_CACHE_DIR)
cache_manifest = RAM_CACHE_DIR / 'ram_cache_manifest.json'
cache_ready = (cache_manifest.exists() and
               json.loads(cache_manifest.read_text(encoding='utf-8')).get('complete') is True)
source_env = ['DDB_DATABASE', 'DDB_TABLE', 'DDB_TRADE_DAYS_DATABASE']
connection_env = ['DDB_HOST', 'DDB_PORT', 'DDB_USER', 'DDB_PASSWORD']
required_env = source_env + ([] if cache_ready else connection_env)
missing = [name for name in required_env if not os.environ.get(name)]
if missing:
    raise EnvironmentError('缺少环境变量: ' + ', '.join(missing))
print('RAM_CACHE_DIR =', RAM_CACHE_DIR)
print('RAM缓存存在   =', cache_ready)
print('环境变量检查通过；敏感值未显示')

## 检查内存与配置

正式加载前确认可用内存接近 700GB，并把配置中的 `prices_are_adjusted` 改为真实状态。

In [ ]:
import os
import yaml

CONFIG_PATH = Path('configs/minute/ppu_ddb_ram.yaml')
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
total_ram = os.sysconf('SC_PHYS_PAGES') * os.sysconf('SC_PAGE_SIZE')
available_ram = os.sysconf('SC_AVPHYS_PAGES') * os.sysconf('SC_PAGE_SIZE')
print('total RAM GB     =', round(total_ram / 1024**3, 1))
print('available RAM GB =', round(available_ram / 1024**3, 1))
print('load_mode        =', config['dataset']['dolphindb']['load_mode'])
print('build_workers    =', config['dataset']['memory']['build_workers'])
print('reward_workers   =', config['dataset']['memory']['workers'])
print('ram cache        =', config['dataset']['memory'].get('ram_cache_enabled', True))
assert config['dataset']['memory']['reward_parallel_backend'] == 'threading'

## 选择运行阶段

首次完整运行保留全部 `True`。如果 GFlowNet 已经训练完成并且 `alpha_pool.csv`、`alpha_factor_matrix.csv.gz` 已生成，将 `RUN_GFLOWNET_TRAINING=False`，直接执行后处理。

In [ ]:
RUN_GFLOWNET_TRAINING = True
RUN_ALPHA_EVAL = True
RUN_LIGHTGBM = True
PACKAGE_RESULTS = True
REBUILD_DAILY_PRICE_IF_MISSING = True

## 开始加载并训练

启动时先检查普通 NumPy 磁盘快照。命中会显示 `[DDBRAMCache] hit` 并跳过DDB连接；未命中才显示 `[DDBRAM]` 从DDB加载，完成后保存快照。无论哪种路径，训练期间都只使用普通RAM数组，不使用MemMap。

In [ ]:
import subprocess
import sys

if RUN_GFLOWNET_TRAINING:
    subprocess.run([
        sys.executable, 'scripts/train_cpu.py',
        '--mode', 'minute', '--config', str(CONFIG_PATH),
    ], check=True)
else:
    print('跳过 GFlowNet 训练，使用已有分钟因子产物')

## 验收分钟训练产物并转换格式

训练结束时已经加载最佳 checkpoint、生成 Alpha Pool，并把分钟表达式日内聚合为日频因子。这里检查日期、重复键和覆盖率，再把压缩 CSV 转成 AlphaEval/LightGBM 使用的 Pickle。

In [ ]:
import pandas as pd
import shutil

outputs = config['outputs']
checkpoint_path = Path(outputs['checkpoint'])
pool_path = Path(outputs['alpha_pool'])
factor_csv_path = Path(outputs['factor_matrix'])
factor_pickle_path = Path(outputs['factor_matrix_pickle'])
daily_price_path = Path(outputs['daily_price'])
legacy_daily_price_path = RAM_CACHE_DIR / 'daily_price.pkl'
if not daily_price_path.exists() and legacy_daily_price_path.exists():
    daily_price_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(legacy_daily_price_path, daily_price_path)
    print('从旧RAM快照迁移日频文件:', daily_price_path)
if not daily_price_path.exists() and REBUILD_DAILY_PRICE_IF_MISSING:
    missing_ddb = [name for name in connection_env + source_env if not os.environ.get(name)]
    if missing_ddb:
        raise EnvironmentError('补建daily_price.pkl需要DDB环境变量: ' + ', '.join(missing_ddb))
    print('缺少独立日频文件；只从DDB聚合日频数据，不重训GFlowNet...')
    subprocess.run([
        sys.executable, 'scripts/export_ddb_daily.py',
        '--config', str(CONFIG_PATH),
    ], check=True)
required = [checkpoint_path, pool_path, factor_csv_path, daily_price_path]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('训练产物不完整: ' + ', '.join(missing))

factors = pd.read_csv(factor_csv_path)
factors['date'] = pd.to_datetime(factors['date']).dt.normalize()
factors['code'] = factors['code'].astype(str)
if factors.duplicated(['date', 'code']).any():
    raise ValueError('因子矩阵存在重复 date/code')
factor_columns = [column for column in factors if column not in ('date', 'code')]
if not factor_columns:
    raise ValueError('因子矩阵没有分钟因子列')
coverage = factors[factor_columns].notna().mean().sort_values()
min_coverage = float(config['reward']['min_coverage'])
low_coverage = coverage[coverage < min_coverage]
if len(low_coverage):
    raise ValueError(f'存在低覆盖因子(<{min_coverage:.0%}): {low_coverage.to_dict()}')
factor_pickle_path.parent.mkdir(parents=True, exist_ok=True)
factors.to_pickle(factor_pickle_path)
print('rows =', len(factors))
print('dates =', factors['date'].nunique())
print('stocks =', factors['code'].nunique())
print('date range =', factors['date'].min().date(), factors['date'].max().date())
print('factors =', len(factor_columns))
print('coverage min/median/max =', round(coverage.min(), 4), round(coverage.median(), 4), round(coverage.max(), 4))
print('pickle saved =', factor_pickle_path)

## 运行 AlphaEval

AlphaEval 只使用 2020–2023 样本内区间，计算 RankIC、ICIR、稳定性、扰动鲁棒性和 DPP 多样性。

In [ ]:
alpha_eval_path = Path(outputs['alpha_eval'])
if RUN_ALPHA_EVAL:
    subprocess.run([
        sys.executable, '-m', 'src.alpha_eval.run_evaluation',
        '--config', str(CONFIG_PATH),
        '--price', str(daily_price_path),
        '--factors', str(factor_pickle_path),
        '--metadata', str(pool_path),
        '--output', str(alpha_eval_path),
    ], check=True)
elif not alpha_eval_path.exists():
    raise FileNotFoundError(f'跳过 AlphaEval，但结果不存在: {alpha_eval_path}')
evaluation = pd.read_csv(alpha_eval_path)
selected = evaluation.loc[evaluation['dpp_selected'].astype(bool), 'factor']
if selected.empty:
    raise ValueError('AlphaEval 没有选中任何因子')
print('evaluated =', len(evaluation), 'selected =', len(selected))
display(evaluation.head(30))

## 运行 LightGBM

使用 AlphaEval 入选因子，带 5 日 purge 做滚动训练，并只输出 2024–2026 的股票预测分数。

In [ ]:
lightgbm_dir = Path(outputs['lightgbm_dir'])
prediction_path = lightgbm_dir / 'prediction_score.csv'
if RUN_LIGHTGBM:
    subprocess.run([
        sys.executable, '-m', 'src.model.run_lightgbm',
        '--config', str(CONFIG_PATH),
        '--price', str(daily_price_path),
        '--factors', str(factor_pickle_path),
        '--evaluation', str(alpha_eval_path),
        '--output-dir', str(lightgbm_dir),
    ], check=True)
elif not prediction_path.exists():
    raise FileNotFoundError(f'跳过 LightGBM，但预测文件不存在: {prediction_path}')
prediction = pd.read_csv(prediction_path)
prediction['signal_date'] = pd.to_datetime(prediction['signal_date'])
print('prediction rows =', len(prediction))
print('prediction dates =', prediction['signal_date'].nunique())
print('prediction range =', prediction['signal_date'].min().date(), prediction['signal_date'].max().date())
display(pd.read_csv(lightgbm_dir / 'model_metrics.csv').tail(20))

## 保存后处理清单并打包

压缩包只保存继续研究和本地回测必需的模型、表达式、评价结果、预测与指标；不会打包数十 GB 的 RAM 分钟快照。

In [ ]:
from datetime import datetime, timezone
import zipfile

artifact_paths = [
    checkpoint_path, pool_path, alpha_eval_path, prediction_path,
    Path(outputs['metrics']), Path(outputs['trajectory_metrics']),
    lightgbm_dir / 'model_metrics.csv',
    lightgbm_dir / 'feature_importance.csv',
    lightgbm_dir / 'lgbm_model.joblib',
]
missing = [str(path) for path in artifact_paths if not path.exists()]
if missing:
    raise FileNotFoundError('后处理产物不完整: ' + ', '.join(missing))
manifest = {
    'pipeline': 'minute_ppu_postprocess',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'training_period': [config['dataset']['mining_start_date'], config['dataset']['mining_end_date']],
    'prediction_period': [config['lightgbm']['prediction_start_date'], config['lightgbm']['prediction_end_date']],
    'factor_rows': len(factors),
    'factors': len(factor_columns),
    'selected_factors': len(selected),
    'prediction_rows': len(prediction),
    'artifacts': [str(path) for path in artifact_paths],
}
manifest_path = Path(outputs['postprocess_manifest'])
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
package_path = Path(outputs['artifact_package'])
if PACKAGE_RESULTS:
    package_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(package_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(CONFIG_PATH, CONFIG_PATH.as_posix())
        archive.write(manifest_path, manifest_path.as_posix())
        for path in artifact_paths:
            archive.write(path, path.as_posix())
    print('package =', package_path, 'MB =', round(package_path.stat().st_size / 1024**2, 1))
print(json.dumps(manifest, ensure_ascii=False, indent=2))

## 下载到本地回测

下载 `results/minute_ppu_ddb_ram/minute_ppu_artifacts.zip`。本地 RQAlphaPlus 只需要其中的 `prediction_score.csv`，不需要下载 RAM 快照或完整因子矩阵。具体命令见 `docs/guides/minute/expression_gflownet.md`。